# 第11回　モデル選択とAIC
## ―― 当てはまりの良さ vs 複雑さ、オッカムの剃刀

統計学Ⅰ（B）　／　北星学園大学

第10回で「変数を足せばR²は上がる」と分かった。じゃあ全部入れればいい？　注目は ――

> 良いモデルとは「**当てはまり**」が良いのではなく、「**まだ見ぬデータをよく予測する**」モデルだ。

### フック

> 第10回で、行動圏を予測するモデルの R² は 0.46 → 0.60 → **0.75** と上がった。
> 
> **これは良いモデルになったのか？　それとも、でたらめを足したときと同じことが起きたのか？**

R² だけを見ていては区別がつかない。今日はその区別のしかたを学ぶ。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
import statsmodels.api as sm

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 0. 第10回のモデルを用意する

In [ ]:
cols = ["行動圏km2", "体重g", "集団サイズ", "個体群密度"]
s = df.dropna(subset=cols).copy()
s = s[(s[cols] > 0).all(axis=1)]
for c in cols:
    s["log" + c] = np.log10(s[c])

y    = s["log行動圏km2"].reset_index(drop=True)
本物 = s[["log体重g", "log集団サイズ", "log個体群密度"]].reset_index(drop=True)

rng = np.random.default_rng(2026)
でたらめ = pd.DataFrame({f"でたらめ{j}": rng.normal(0, 1, len(s)) for j in range(20)})

print(f"使う種数 n = {len(s)}")
print(f"本物の説明変数 : {list(本物.columns)}")
print(f"でたらめ列     : 20本（行動圏とは何の関係もない乱数）")

---
## 1. 過学習を、目で見る

データを **訓練用（7割）** と **テスト用（3割）** に分ける。訓練用だけでモデルを作り、**見せていないテスト用**でどれだけ予測できるかを測る。

本物の変数に「でたらめ列」を足していきながら、

- **訓練R²**（手元への当てはまり）
- **テスト誤差 RMSE**（未知データの予測のズレ。小さいほど良い）

の両方を追う。

> **分割は1回だと運に左右される。**第6回でやったとおり、**200回くりかえして平均**する。1回の結果で判断しないのが今日の作法でもある。

In [ ]:
n = len(s); cut = int(n * 0.7)
ks = [0, 2, 5, 10, 15, 20]
訓練R2, テストRMSE = [], []

print(f"訓練 {cut} 種 / テスト {n-cut} 種　（分割を200回くりかえして平均）\n")
print("  でたらめ    訓練R²   テストRMSE")
for k in ks:
    X = pd.concat([本物, でたらめ.iloc[:, :k]], axis=1)
    r2s, rmses = [], []
    for _ in range(200):
        idx = rng.permutation(n); tr, te = idx[:cut], idx[cut:]
        m = sm.OLS(y.iloc[tr], sm.add_constant(X.iloc[tr])).fit()
        Xte = sm.add_constant(X.iloc[te], has_constant="add")
        r2s.append(m.rsquared)
        rmses.append(np.sqrt(np.mean((y.iloc[te] - m.predict(Xte))**2)))
    訓練R2.append(np.mean(r2s)); テストRMSE.append(np.mean(rmses))
    print(f"  {k:>6}本   {np.mean(r2s):.3f}    {np.mean(rmses):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ks, 訓練R2, "o-", color="#00897b")
ax[0].set_title("訓練R²（手元への当てはまり）")
ax[0].set_xlabel("でたらめ列の数"); ax[0].set_ylabel("R²")
ax[1].plot(ks, テストRMSE, "o-", color="#e8503a")
ax[1].set_title("テスト誤差RMSE（未知データの予測）")
ax[1].set_xlabel("でたらめ列の数"); ax[1].set_ylabel("RMSE（小さいほど良い）")
plt.suptitle("当てはまりは上がり続けるのに、予測は悪化していく＝過学習")
plt.tight_layout(); plt.show()

**訓練R²は 0.748 → 0.809 と上がり続ける**（手元に合わせるのが上手くなる）のに、**テスト誤差は 0.474 → 0.541 と悪化していく**。

これが **過学習**：手元のデータに合わせすぎて、未知のデータに弱くなる。

つまり「当てはまりの良さ（R²）」と「予測の良さ」は別物。**R²を最大化してはいけない。**

> 今回は n=137種しかない。**データが少ないほど過学習は激しい。**> 20本のでたらめを足すと、パラメータ数は23個。137種で23個のパラメータを推定すれば、> ノイズまで覚え込んでしまう。

---
## 2. AIC ―― テストデータ無しで「予測の良さ」を見積もる

毎回データを分けるのは大変（しかも200回もやった）。**AIC（赤池情報量規準）** は、**手元のデータだけ**から「予測の良さ」を見積もる指標だ。

$$ \text{AIC} = \underbrace{-2 \times (\text{当てはまりの良さ})}_{\text{小さいほど当てはまり良}} + \underbrace{2 \times (\text{パラメータの数})}_{\text{複雑さのペナルティ}} $$

- 当てはまりが良いほど第1項が小さくなる（AIC下がる）
- 変数を増やすほど第2項が大きくなる（AIC上がる＝ペナルティ）
- **AICが小さいモデルほど良い**。＝「当てはまり」と「複雑さ」のバランスが最良。

これは **オッカムの剃刀**（同じ説明力なら単純なほうを選べ）の定量化だ。

第10回で作った3つのモデルで比べてみよう。

In [ ]:
全変数 = pd.concat([本物, でたらめ], axis=1)

モデル = {
    "M1 体重のみ":            ["log体重g"],
    "M2 体重+集団+密度":      ["log体重g", "log集団サイズ", "log個体群密度"],
    "M3 M2+でたらめ10本":     ["log体重g", "log集団サイズ", "log個体群密度"]
                              + [f"でたらめ{j}" for j in range(10)],
}

print(f"{'モデル':<24}{'変数の数':>8}{'R²':>10}{'AIC':>12}")
print("-" * 56)
結果 = {}
for name, use in モデル.items():
    m = sm.OLS(y, sm.add_constant(全変数[use])).fit()
    結果[name] = (m.rsquared, m.aic)
    print(f"{name:<24}{len(use):>8}{m.rsquared:>10.3f}{m.aic:>12.1f}")

best_r2  = max(結果, key=lambda k: 結果[k][0])
best_aic = min(結果, key=lambda k: 結果[k][1])
print()
print(f"R²が最大  → {best_r2}")
print(f"AICが最小 → {best_aic}   ← 採用すべきはこちら")

**R²が最大なのは M3（でたらめ入り、0.775）。でもAICが最小＝採用すべきは M2（184.2）。**両者は一致しない！

| モデル | 変数 | R² | AIC |
|---|---:|---:|---:|
| M1 体重のみ | 1 | 0.468 | 281.4 |
| M2 体重+集団+密度 | 3 | 0.746 | **184.2（最小）** |
| M3 M2+でたらめ10本 | 13 | **0.775（最大）** | 187.7 |

- **M1→M2**：本物の変数（集団サイズ・個体群密度）を足したらAICは 281 → 184 と大きく下がった（＝役に立つ変数は歓迎）。
- **M2→M3**：でたらめを足したらR²は上がったがAICは 184 → 188 と上がった（＝中身のない複雑さは罰する）。

AICは「R²の罠」に引っかからず、過学習しないモデルを選んでくれる。

> **第10回のフックへの答え。**
> 「R²が0.75まで上がったのは良いモデルになったのか」――**なっていた。**
> M2 は AIC でも最小である。集団サイズと個体群密度は、本当に行動圏の情報を持っていた。
> だが**それを R² だけで判断することはできなかった**。AICという別の物差しが要る。

---
## 3. AICの使い方の作法

- **AICは相対指標**。**絶対値そのものに意味はない**（184.2という数字単体は何も語らない）。**モデル間の差**で比べる。
- 差が大きいほど優劣は明確（目安：差が2以上で意味あり、10以上で決定的）。今回の M2 と M3 の差は 3.5 なので「M2のほうが良い」と言える程度。M1 と M2 の差は 97 で、こちらは決定的。
- 比べるモデルは**同じデータ・同じ目的変数**で作ること。**欠測で行数が変わると比較できない**（今日はあらかじめ137種に揃えてある）。
- 「AICが小さい」＝「予測が良い」の見積もりであって、**真のモデルを当てる保証ではない**。あくまで候補の中での相対的なベスト。

> ❌ よくある誤り：「AICの絶対値が小さい/大きいことに意味がある」「変数は多いほど良い」「AICが最小なら正しいモデル」。

!!! 注意
    第5回で見たとおり、生き物のデータでは**体の大きさが何にでも効く**。AICが「この変数を入れよ」と言ったからといって、**それが原因だという意味ではない**。モデル選択は予測の話であって、因果の話ではない。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 過学習 | 手元に合わせすぎて未知データに弱くなる（訓練R²↑なのにテスト誤差↑） |
| 当てはまり vs 予測 | R²最大化はダメ。狙うのは未知データの予測 |
| AIC | −2×当てはまり ＋ 2×パラメータ数。**小さいほど良い** |
| オッカムの剃刀 | 同じ説明力なら単純なモデルを選ぶ。AICはその定量化 |
| 差の読み方 | 2以上で意味あり、10以上で決定的。**絶対値には意味がない** |
| ❌ 誤り | AICの絶対値に意味がある／変数は多いほど良い／AIC最小＝因果 |

> **R²最大のモデルとAIC最小のモデルは違う。**
> 良いモデルは「当てはまり」ではなく「当てはまりと複雑さのバランス」で選ぶ。

**課題（Moodle）**：複数モデルのAICを比較して、採用すべきモデルと理由を述べる。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。